In [2]:
import networkx as nx
import numpy as np
import pandas as pd

np.random.seed(42)
G = nx.erdos_renyi_graph(50, 0.15, seed=42)
for u, v in G.edges():
    G[u][v]['latency'] = round(np.random.uniform(1, 10), 2)

# Pre-compute betweenness
betweenness = nx.betweenness_centrality(G)

def bcr_route(G, src, dst, centrality):
    """Route packet using Betweenness Centrality Routing."""
    if src == dst:
        return [src], 0
    path = [src]
    current = src
    visited = set([src])
    max_hops = 50  # prevent infinite loop

    for _ in range(max_hops):
        neighbors = [n for n in G.neighbors(current) if n not in visited]
        if not neighbors:
            return None, float('inf')  # dead end
        # Pick neighbour with highest betweenness
        next_hop = max(neighbors, key=lambda n: centrality[n])
        path.append(next_hop)
        visited.add(next_hop)
        if next_hop == dst:
            return path, len(path) - 1
        current = next_hop

    return None, float('inf')

# Simulate 100 packet deliveries
results = []
nodes = list(G.nodes())

for i in range(100):
    src = np.random.choice(nodes)
    dst = np.random.choice([n for n in nodes if n != src])
    path, hops = bcr_route(G, src, dst, betweenness)
    delivered = path is not None
    delay = sum(G[path[j]][path[j+1]]['latency']
                for j in range(len(path)-1)) if delivered else 0
    results.append({'method':'BCR','src':src,'dst':dst,
                    'delivered':delivered,'hops':hops,'delay':delay})

df = pd.DataFrame(results)
pdr  = df['delivered'].mean() * 100
avgd = df[df['delivered']]['delay'].mean()
avgh = df[df['delivered']]['hops'].mean()

print(f"BCR — PDR: {pdr:.2f}%  |  Avg Delay: {avgd:.2f} ms  |  Avg Hops: {avgh:.2f}")
df.to_csv("../results/bcr_results.csv", index=False)

BCR — PDR: 65.00%  |  Avg Delay: 101.26 ms  |  Avg Hops: 18.05
